In [1]:
import sys
import torch
import pandas as pd
from config import SimulationConfig, PersonalityConfig, ModelConfig
from questionbank import QuestionBank
from retriever import RetrieverModel, QuestionRetriever
from storememory import MemoryStore
from timestamp import TimestampManager
from models.teacheragent import TeacherAgent
from models.studentlearning import StudentLearningAgent
from models.studentexam import StudentExamAgent
from environments.learningloop import LearningLoop
from environments.examloop import ExamLoop
from evaluation.metrics import AnswerEvaluator
from main import run_simulation, set_seed

print("All modules imported successfully")
print(f"CUDA available: {torch.cuda.is_available()}")

All modules imported successfully
CUDA available: True


In [ ]:
RANDOM_SEED = 42
EXAM_TOPIC = "Algebra" #Algebra, Geometry, Number Theory, Counting & Probability
MODEL_TYPE = "api"  # or "local"
LEARNING_ROUNDS = 5
NUM_QUESTIONS = 50

personalities = [
    ('high_openness', PersonalityConfig.get_high_openness()),
    ('high_conscientiousness', PersonalityConfig.get_high_conscientiousness()),
    ('high_extraversion', PersonalityConfig.get_high_extraversion()),
    ('high_agreeableness', PersonalityConfig.get_high_agreeableness()),
    ('high_neuroticism', PersonalityConfig.get_high_neuroticism()),
]

print("="*80)
print(f"BATCH EXPERIMENT: Running {len(personalities)} personalities")
print(f"Topic: {EXAM_TOPIC}")
print(f"Learning rounds: {LEARNING_ROUNDS}")
print(f"Exam questions: {NUM_QUESTIONS}")
print(f"Model type: {MODEL_TYPE}")
print("="*80)

all_results = []

for idx, (name, personality) in enumerate(personalities, 1):
    print(f"\n{'#'*80}")
    print(f"# Experiment {idx}/{len(personalities)}: {name}")
    print(f"{'#'*80}\n")
    
    config = SimulationConfig(
        random_seed=RANDOM_SEED,
        exam_topic=EXAM_TOPIC,
        model_config=ModelConfig(model_type=MODEL_TYPE)
    )
    
    config.learning_config.learning_rounds = LEARNING_ROUNDS
    config.exam_config.num_questions = NUM_QUESTIONS
    config.personality = personality
    
    try:
        result = run_simulation(config, personality)
        
        all_results.append({
            'personality': name,
            'accuracy': result['exam_summary']['accuracy'],
            'micro_f1': result['metrics']['micro_f1'],
            'macro_f1': result['metrics']['macro_f1'],
            'empty_percentage': result['exam_summary']['empty_answer_percentage'],
            'correct_count': result['exam_summary']['correct_count'],
            'self_study': result['learning_summary']['action_counts']['self_study'],
            'ask_teacher': result['learning_summary']['action_counts']['ask_teacher'],
            'skip': result['learning_summary']['action_counts']['skip'],
            'learning_time': result['learning_summary']['final_timestamp'],
            'exam_time': result['exam_summary']['final_timestamp']
        })
        
        print(f"\n✓ {name} completed successfully")
        
    except Exception as e:
        print(f"\n✗ {name} failed: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*80)
print("BATCH EXPERIMENT SUMMARY")
print("="*80)

if all_results:
    df_results = pd.DataFrame(all_results)
    print("\n" + df_results.to_string(index=False))
    
    print("\n" + "="*80)
    print("KEY FINDINGS")
    print("="*80)
    
    best_accuracy = df_results.loc[df_results['accuracy'].idxmax()]
    best_f1 = df_results.loc[df_results['macro_f1'].idxmax()]
    least_empty = df_results.loc[df_results['empty_percentage'].idxmin()]
    
    print(f"\nBest Accuracy: {best_accuracy['personality']} ({best_accuracy['accuracy']:.2%})")
    print(f"Best Macro F1: {best_f1['personality']} ({best_f1['macro_f1']:.4f})")
    print(f"Least Empty Answers: {least_empty['personality']} ({least_empty['empty_percentage']:.1f}%)")
#accuracy calculation has some error, it's just for reference, use macro F1 as the final result
    comparison_file = f"results/personality_comparison_lr{LEARNING_ROUNDS}.csv"
    df_results.to_csv(comparison_file, index=False)
    print(f"\nComparison saved to: {comparison_file}")
else:
    print("\nNo results collected. Check errors above.")